# Clef-Flash on Colab / Kaggle → a Plexus `decisions` upstream — **v10**

> **v10 runs beside other notebooks.** Each runtime gets its own UUID, provider slug, Tailscale
> hostname, private daemon state and bearer, and publishes itself through a tailnet-only
> **registrar** that owns a dedicated pool alias. Two v10 runtimes never write the same object.

**What changed from v9.** v9 rewrote the *shared* provider `clef-llama-remote` and toggled a rung
in `cloudflare/clef-flash`'s `LOCAL` group, so a second notebook overwrote the first, and startup
deleted same-named Tailscale nodes it had not proven were its own. v10 writes nothing shared: no
parent alias, no provider another run owns, and no device it cannot prove is its own.

Unchanged from v9, deliberately: the GGUF (`bartowski/Cloudflare_clef-flash-GGUF`, Q4_K_M), the
llama.cpp release (`b12292`), the CUDA runtime, `-b 16384 -ub 16384`, `CTX_T4` 32,768, the
GPU-residency proof, and the userspace outbound HTTP proxy that is the only way off this VM.

**What a run does**

1. serves `POST /v1/systemone` from llama.cpp `b12292` behind a round-robin, bearer-checking proxy;
2. joins the tailnet as an **ephemeral, preauthorised** node named `clef-nb-<uuid>` — a name no
   other runtime can have — on a private state file and socket under `runs/<uuid>/state`;
3. takes a **VM-wide lock** first: one runtime per VM, because a second kernel would contend for
   the GPUs, the loopback worker ports and tailscaled's userspace proxy port. **Separate Colab or
   Kaggle runtimes are independent and supported**; two kernels in one VM are refused;
4. creates a **lease** carrying its identity, worker count and per-run bearer;
5. asks the registrar for **admission**. The registrar owns this run's provider
   `clef-nb-<uuid>`, probes it *through Plexus* with a temporary provider-scoped key, and only
   then enables that single target in the dedicated `clef-flash-notebooks` pool. A notebook
   cannot admit itself, and there is no switch that skips the proof;
6. renews the lease every 20 s. The lease TTL is 180 s; expiry or withdrawal is terminal for that
   UUID, and the registrar keeps the provider as a disabled tombstone so the slug is never reused.

**Teardown** withdraws the lease and verifies it, waits for the routing reload and for local
in-flight requests to drain (600 s cap), then stops workers, proxy, and this run's own Tailscale
node. If withdrawal cannot be confirmed, everything keeps running and teardown reports INCOMPLETE
— a routed dead endpoint is worse than a live unused one.

**Hard death.** If the VM is deleted, no callback runs. The registrar expires the lease within
180 s and disables that one target; nobody has to edit the pool by hand.

**Standalone pool vs parent integration.** `clef-flash-notebooks` is a decisions pool that is
usable on its own — call the alias directly. Inserting a pointer to it into
`cloudflare/clef-flash` / `clef-flash-local` is a **one-time operator action** taken in a
quiescent control window. Notebook start, heartbeat, expiry and teardown never write a parent
alias.

## Hardware and session limits

| Requirement | Detail |
|---|---|
| GPU runtime | **required** — `Runtime → Change runtime type → T4 GPU` |
| GPUs | 2× T4 recommended (two workers); 1× T4 works (one worker); VRAM < 14 GiB is refused |
| One runtime per VM | a second kernel in the same VM is refused; separate VMs are independent |
| Download | ~8 GB (llama.cpp 198 MB + CUDA runtime 594 MB + GGUF 6.04 GB) into `/content` |
| Session cap | ~12 h on Colab Pro; free tier is shorter |
| Idle | ~90 min with no browser tab → disconnect; closing the tab ends the VM |

Everything mutable lives in the VM's `/content`, which is wiped with the VM. Model and binary
caches stay under `ROOT` and are re-used across runs on the same VM; per-run logs, the tailscaled
state and the runtime lock live under `ROOT/runs/<uuid>/`.

## Context per GPU (KV cache: 32 KiB/token)

| VRAM | `-c` | KV (f16) | Notes |
|---|---|---|---|
| ≥ 22 GiB | 262144 | 8.00 GiB | full trained context |
| 14–22 GiB (T4) | 32768 | 1.00 GiB | v9 T4 shape: 6.04 GB weights + KV + the 16,384-token physical batch |
| < 14 GiB | — | — | the notebook refuses to start |

Override with `CLEF_CTX`; hard cap 262144. The value actually in force is read back from
`GET /props` → `default_generation_settings.n_ctx` and printed per worker.


## Secrets (Colab key panel, Kaggle private dataset, or the environment outside a notebook)

Two secrets, and no Plexus credential at all:

| Name | Required | Used for |
|---|---|---|
| `TAILSCALE_API_KEY` | yes | Tailscale REST API: mint this run's single-use ephemeral auth key, resolve this run's device id, delete only that device on teardown |
| `CLEF_REGISTRY_KEY` | yes | bearer for the registrar: create/renew/admit/withdraw this run's lease |

`CLEF_REGISTRY_URL` is **not** a secret: it is the tailnet URL of the registrar, supplied as an
environment variable (or a `.env` file) and validated for shape before use. There is no default —
a published notebook must not carry a household endpoint.

What the notebook does *not* hold, and cannot use: `PLEXUS_ADMIN_KEY`, `PLEXUS_API_KEY`, or any
shared provider key. The per-run bearer that Plexus sends (`CLEF_REMOTE_KEY`) is generated in this
kernel with `secrets.token_hex(32)`, handed to the registrar once at lease creation, and never
rotated afterwards.

Cells read `google.colab.userdata` and fall back to `os.environ`, so the same code runs outside
Colab. **No secret value is ever printed** — only names and lengths.

The bearer lands in the `llama-server` command line, so it is visible to `ps` inside the VM.
Acceptable on a single-tenant ephemeral VM; not acceptable on a shared host.


In [ ]:
%%bash
# [shell 1/3] python deps. torch/CUDA already ship with a GPU runtime; do not reinstall them.
set -euo pipefail
python3 -m pip install --quiet --upgrade "huggingface_hub>=0.23.0"
python3 - <<'PY'
import sys, huggingface_hub
print("python         ", sys.version.split()[0])
print("huggingface_hub", huggingface_hub.__version__)
try:
    import torch
    print("torch          ", torch.__version__, "cuda available:", torch.cuda.is_available())
except Exception as exc:
    print("torch           not importable:", exc)
PY

In [ ]:
# [2/13] configuration: secrets, paths, constants
from __future__ import annotations

import ipaddress
import os
import secrets
import subprocess
import time
import urllib.parse
from pathlib import Path


# ------------------------------------------------------------------ .env file
# A `.env` file can carry the four keys instead of the Colab secret panel: read it
# BEFORE the secrets block so `_secret()` finds them in os.environ.  Never log a
# value -- only the path that loaded and the KEY NAMES it supplied.
def _load_env_file() -> str:
    candidates = []
    explicit = os.environ.get("CLEF_ENV_FILE", "").strip()
    if explicit:
        candidates.append(Path(explicit))
    candidates += [
        Path("/content/.env"),                    # Colab upload / %%writefile
        Path("/content/drive/MyDrive/.env"),      # Colab + mounted Drive
        Path("/kaggle/working/.env"),             # Kaggle, written by the kernel
        Path.cwd() / ".env",
    ]
    for path in candidates:
        try:
            if not path.is_file():
                continue
            text = path.read_text(errors="replace")
        except Exception:
            continue
        loaded = []
        for raw in text.splitlines():
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export "):].lstrip()
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            if not key or not key.replace("_", "").isalnum():
                continue
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
                value = value[1:-1]
            if os.environ.get(key):
                continue                      # real environment wins over the file
            os.environ[key] = value
            loaded.append(key)
        print(f"[env] {path} -> {len(loaded)} key(s): {', '.join(loaded) or '-'}", flush=True)
        return str(path)
    return ""


ENV_FILE = _load_env_file()

# --------------------------------------------------------------------- secrets
def _secret(name: str, required: bool = True) -> str:
    """Colab userdata first, environment second. Never echoed."""
    value = ""
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        try:
            value = userdata.get(name) or ""
        except Exception:
            value = ""
    except Exception:
        value = ""
    if not value:
        value = os.environ.get(name, "")
    value = (value or "").strip()
    if required and not value:
        raise SystemExit(
            f"missing secret {name}: add it in the Colab key panel, or export it when running locally"
        )
    return value


# Kaggle: mount the private secrets dataset under /kaggle/input.  Values are
# never printed; only the path that supplied them is logged.
try:
    import json as _json
    _sf = None
    for _p in Path("/kaggle/input").rglob("*.json"):
        try:
            _d = _json.loads(_p.read_text())
        except Exception:
            continue
        if {"TAILSCALE_API_KEY", "CLEF_REGISTRY_KEY"} <= set(_d):
            _sf = _p
            for _k, _v in _d.items():
                os.environ.setdefault(_k, str(_v))
            break
    if _sf is not None:
        print(f"[secrets] mounted {_sf}")
    else:
        print("[secrets] no Kaggle secrets dataset found; falling back to env/Colab")
except Exception as _e:
    print(f"[secrets] mount scan failed: {_e}")


# =====================================================================================
# Embedded lease client -- generated verbatim from clef_lease.py by tools/build_v10.py.
# Edit clef_lease.py and regenerate this notebook; do not edit this block here.
# =====================================================================================
"""Per-runtime lease client for the Clef notebook pool.

Embedded verbatim into one notebook code cell by ``tools/build_v10.py``.
Standard library only, self-contained, no import-time side effects.
"""

import json
import secrets
import threading
import time
import urllib.error
import urllib.request
import uuid

WINDOW = 120.0      # seconds without a confirmed renewal before the lease counts as lost
ATTEMPTS = 3        # create/withdraw attempts, including the first
TERMINAL = (400, 403, 404, 409, 410)   # never retried, never resurrected
PHASES = ('staging', 'admitting', 'active', 'expired', 'withdrawn')
UNRECOGNISED = 'unrecognised-phase'


def identity():
    """A fresh runtime identity: unique run id, 256-bit lease token, node hostname."""
    run = uuid.uuid4().hex
    return {'run_id': run, 'token': secrets.token_hex(32), 'hostname': 'clef-nb-' + run}


class LeaseFault(RuntimeError):
    """A lease operation failure. Never carries a credential, a body or a header."""

    def __init__(self, status, reason):
        self.status = status
        self.reason = reason
        super().__init__(reason if status is None else '%s (status=%s)' % (reason, status))


class Lease:
    """Client for one runtime's registrar lease.

    ``opener`` is a zero-argument factory returning an object with
    ``.open(request, timeout=...)``; ``interval``/``now`` are injectable so tests
    never wait on wall-clock time.
    """

    def __init__(self, base, key, ident, opener, healthy,
                 interval=20.0, now=time.monotonic):
        self.base = base.rstrip('/')
        self.key = key
        self.ident = dict(ident)
        self.opener = opener
        self.healthy = healthy
        self.interval = float(interval)
        self.now = now
        self.halt = threading.Event()
        self.admit_lock = threading.Lock()
        self.thread = None
        self.admit_thread = None
        self.admitting = False
        self.lost = False
        self.withdrawn = False
        self.phase = None
        self.last_error = None
        self.last_confirmed = now()

    def __repr__(self):
        return ('Lease(run_id=%s, phase=%s, lost=%s, withdrawn=%s)'
                % (self.ident['run_id'], self.phase, self.lost, self.withdrawn))

    __str__ = __repr__

    # -- transport ---------------------------------------------------------

    def _url(self, suffix=''):
        return self.base + '/v1/leases/' + self.ident['run_id'] + suffix

    @staticmethod
    def _phase_text(phase):
        """Registrar text is untrusted: only known phases may reach a diagnostic."""
        return phase if phase in PHASES else UNRECOGNISED

    def _reason(self, method, suffix, detail):
        return 'registrar %s %s for run %s failed: %s' % (
            method, suffix or '/status', self.ident['run_id'], detail)

    def call(self, method, suffix='', body=None, timeout=30):
        """One HTTP attempt. Raises LeaseFault, never a raw urllib exception."""
        data = None if body is None else json.dumps(body).encode()
        headers = {'authorization': 'Bearer ' + self.key,
                   'x-lease-token': self.ident['token']}
        if data is not None:
            headers['content-type'] = 'application/json'
            headers['content-length'] = str(len(data))
        request = urllib.request.Request(self._url(suffix), data=data,
                                        headers=headers, method=method)
        try:
            with self.opener().open(request, timeout=timeout) as response:
                raw = response.read()
        except urllib.error.HTTPError as exc:
            code = getattr(exc, 'code', None)
            try:
                exc.close()     # neither the body nor the socket may be retained
            except Exception:
                pass
            raise LeaseFault(code, self._reason(method, suffix, 'status %s' % code))
        except Exception as exc:
            raise LeaseFault(None, self._reason(method, suffix,
                                                'transport ' + type(exc).__name__))
        try:
            payload = json.loads(raw) if raw else {}
        except (TypeError, ValueError):
            raise LeaseFault(None, self._reason(method, suffix, 'undecodable response'))
        if not isinstance(payload, dict):
            raise LeaseFault(None, self._reason(method, suffix, 'unexpected response shape'))
        return payload

    # -- write with bounded, idempotent retries -----------------------------

    def _write(self, method, suffix, body, label, verify, attempts=ATTEMPTS, timeout=30):
        """Bounded retries that reuse the same run id, token and body.

        503 means "not confirmed, retry safely"; a transport failure is ambiguous
        and is resolved by an authenticated status read before any retry.
        400/403/404/409/410 are never retried and never resurrect a lease.
        """
        last = None
        for _ in range(attempts):
            try:
                payload = self.call(method, suffix, body, timeout=timeout)
                self.last_error = None
                return payload
            except LeaseFault as exc:
                last = exc
                self.last_error = exc.reason
                if exc.status is not None and exc.status != 503:
                    raise
                if exc.status == 503:
                    continue
                resolved = verify()
                if resolved is not None:
                    self.last_error = None
                    return resolved
        reason = '%s not confirmed after %d attempts for run %s' % (
            label, attempts, self.ident['run_id'])
        self.last_error = reason
        raise LeaseFault(None if last is None else last.status, reason)

    def _verify_withdrawn(self):
        try:
            payload = self.call('GET')
        except LeaseFault as exc:
            if exc.status is None:
                return None
            raise
        if payload.get('phase') == 'withdrawn':
            return payload
        return None             # still live: repeat the identical, idempotent DELETE

    def _verify_created(self):
        try:
            return self.call('GET')
        except LeaseFault as exc:
            if exc.status is None or exc.status == 404:
                return None     # not applied (or still unknown): repeat the identical PUT
            raise

    # -- lifecycle ---------------------------------------------------------

    def create(self, base, workers, bearer):
        if self.thread is not None and self.thread.is_alive():
            raise RuntimeError('lease already has a renewal thread for run '
                               + self.ident['run_id'])
        payload = self._write('PUT', '', {'base': base, 'workers': workers,
                                         'bearer': bearer},
                             'registration', self._verify_created)
        self.phase = self._phase_text(payload.get('phase'))
        self.last_confirmed = self.now()
        self._start_watch()
        return payload

    def status(self):
        """Authenticated status read for diagnostics.

        Read-only by construction: it reports the registrar's phase and never claims a
        withdrawal (only a confirmed DELETE may do that) and never re-creates a lease.
        """
        payload = self.call('GET')
        self.phase = self._phase_text(payload.get('phase'))
        return payload

    def _start_watch(self):
        """Start the single dedicated renewal thread for this run id."""
        if self.thread is not None and self.thread.is_alive():
            raise RuntimeError('lease already has a renewal thread for run '
                               + self.ident['run_id'])
        self.thread = threading.Thread(target=self._watch, name='clef-lease-renew',
                                       daemon=True)
        self.thread.start()

    def _watch(self):
        while not self.halt.wait(self.interval):
            self._renew()

    def _renew(self):
        try:
            workers = self.healthy()
        except Exception as exc:
            self._unconfirmed('healthy worker count unavailable (%s)'
                              % type(exc).__name__)
            return
        try:
            payload = self.call('POST', '/heartbeat', {'workers': workers})
        except LeaseFault as exc:
            if exc.status == 410:
                # Terminal: the registrar tombstoned this lease. Never resurrect it.
                self.lost = True
                self.last_error = exc.reason
                self.halt.set()
                return
            self._unconfirmed(exc.reason)
            return
        phase = self._phase_text(payload.get('phase'))
        self.phase = phase
        if phase in ('withdrawn', 'expired'):
            self.lost = True
            self.last_error = 'lease ended: phase %s for run %s' % (
                phase, self.ident['run_id'])
            self.halt.set()
            return
        self.last_confirmed = self.now()
        # Confirmation returned, so this runtime is no longer in an unconfirmed window. Leaving
        # `lost` sticky would strand a runtime that outlived a transient outage: it would keep
        # serving while the registrar's demotion kept it out of the pool, forever.
        self.lost = False
        self.last_error = None
        if phase == 'staging':
            # Registrar restarted or demoted us: re-prove admission in the background
            # while this renewal thread keeps the lease alive.
            self._start_admission()

    def _start_admission(self):
        """Start at most one background admission worker for this lease."""
        with self.admit_lock:
            if self.admitting or self.withdrawn or self.lost:
                return False
            self.admitting = True
            self.admit_thread = threading.Thread(
                target=self._admit, name='clef-lease-admit', daemon=True)
            self.admit_thread.start()
            return True

    def _admit(self):
        try:
            self.ready()
        except Exception as exc:
            reason = getattr(exc, 'reason', None) or type(exc).__name__
            self.last_error = 're-admission failed: %s' % reason
        finally:
            with self.admit_lock:
                self.admitting = False

    def _unconfirmed(self, reason):
        """Record loss of confirmation without ever reporting a withdrawal."""
        self.last_error = reason
        if self.now() - self.last_confirmed > WINDOW:
            self.lost = True

    def withdraw(self):
        """Withdraw this runtime. ``withdrawn`` only becomes true on a confirmed DELETE."""
        payload = self._write('DELETE', '', None, 'withdrawal', self._verify_withdrawn)
        phase = self._phase_text(payload.get('phase'))
        if phase != 'withdrawn':
            self.last_error = 'withdrawal not confirmed; phase %s for run %s' % (
                phase, self.ident['run_id'])
            raise LeaseFault(None, self.last_error)
        self.withdrawn = True
        self.phase = 'withdrawn'
        self.last_error = None
        self.halt.set()
        thread = self.thread
        if thread is not None and thread is not threading.current_thread():
            thread.join(timeout=35)
        return payload

    # -- admission ---------------------------------------------------------

    def ready(self, timeout=650):
        """Ask the registrar to prove and publish this runtime into the pool."""
        try:
            payload = self.call('POST', '/ready', {}, timeout=timeout)
        except LeaseFault as exc:
            self.last_error = exc.reason
            if exc.status == 410:
                self.lost = True        # terminal: the lease is gone, never resurrect
                raise
            if exc.status is not None:
                raise
            payload = self._resolve_ready()
        return self._admitted(payload)

    def _resolve_ready(self):
        """Resolve a lost ready() response by status read, never by a second proof."""
        for _ in range(ATTEMPTS):
            try:
                return self.call('GET')
            except LeaseFault as exc:
                if exc.status is not None:
                    raise
        reason = 'admission outcome unknown after a lost response for run %s' % (
            self.ident['run_id'])
        self.last_error = reason
        raise LeaseFault(None, reason)

    def _admitted(self, payload):
        if self.withdrawn:
            # Stop already won: a late probe result must never re-publish this runtime.
            self.last_error = 'lease already withdrawn; ignoring a late admission'
            raise LeaseFault(None, self.last_error)
        if payload.get('run_id') != self.ident['run_id']:
            self.last_error = 'admission response belongs to another runtime'
            raise LeaseFault(None, self.last_error)
        phase = self._phase_text(payload.get('phase'))
        self.phase = phase
        if phase != 'active':
            self.last_error = 'admission not confirmed; phase %s for run %s' % (
                phase, self.ident['run_id'])
            raise LeaseFault(None, self.last_error)
        self.lost = False
        self.last_error = None
        return payload


# --------------------------------------------------------------- this runtime's identity
import fcntl

_prior_runtime = globals().get("RUNTIME")
if _prior_runtime is not None and not getattr(_prior_runtime, "_stopped", False):
    raise RuntimeError(
        "a live Runtime already exists in this kernel — run its shutdown cell ([13/13]) before "
        "reconfiguring: rebuilding these globals would orphan its workers, proxy and lease"
    )

IDENT = identity()
RUN_ID = IDENT["run_id"]


def _registry_url_ok(value: str) -> bool:
    """A literal tailnet IPv4, with an explicit port, and nothing else.

    The lease token travels in a header over plain http inside the tailnet. A hostname, a public
    address, embedded credentials, a path or a query would let a typo — or a hostile peer that a
    name resolves to — capture this run's credentials, so the shape is checked before any request.
    """
    parsed = urllib.parse.urlsplit(value or "")
    try:
        return (
            parsed.scheme == "http"
            and parsed.path in ("", "/")
            and not parsed.query
            and not parsed.fragment
            and not parsed.username
            and not parsed.password
            and parsed.port is not None
            and ipaddress.ip_address(parsed.hostname) in ipaddress.ip_network("100.64.0.0/10")
        )
    except (TypeError, ValueError):
        return False


REGISTRY_URL = os.environ.get("CLEF_REGISTRY_URL", "").rstrip("/")
if not REGISTRY_URL:
    raise SystemExit(
        "set CLEF_REGISTRY_URL to the registrar's tailnet URL, for example "
        "http://100.x.y.z:8769"
    )
if not _registry_url_ok(REGISTRY_URL):
    raise SystemExit(
        f"CLEF_REGISTRY_URL={REGISTRY_URL!r} is not http://<tailnet IPv4>:<port>; no path, "
        "query, credentials or hostname is accepted"
    )

SECRETS = {
    "TAILSCALE_API_KEY": _secret("TAILSCALE_API_KEY"),
    "CLEF_REGISTRY_KEY": _secret("CLEF_REGISTRY_KEY"),
}
# The bearer Plexus will send is this run's alone. The registrar stores it as the provider's
# api_key at lease creation; it is never adopted from, nor written back to, any shared provider.
SECRETS["CLEF_REMOTE_KEY"] = secrets.token_hex(32)
print("secrets:")
for _name, _value in SECRETS.items():
    print(f"  {_name:<17} {'set  ' if _value else 'unset'} len={len(_value)}")


# ----------------------------------------------------------------------- paths
def _pick_root() -> Path:
    if Path("/kaggle").is_dir():
        kaggle_root = Path("/kaggle/temp/clef")
        kaggle_root.mkdir(parents=True, exist_ok=True)
        return kaggle_root
    content = Path("/content")
    try:
        content.mkdir(parents=True, exist_ok=True)
    except Exception:
        pass
    if content.is_dir() and os.access(content, os.W_OK):
        return content
    fallback = Path(os.path.expanduser("~/.clef-colab"))
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


ROOT = Path(os.environ.get("CLEF_ROOT") or _pick_root())
# Model and binary caches stay process-wide (a 6 GB GGUF and a 200 MB release are re-used), but
# every mutable runtime file lives under runs/<RUN_ID> so two runtimes can never collide.
RUN_ROOT = ROOT / "runs" / RUN_ID
LOG_DIR = RUN_ROOT / "logs"
MODEL_DIR = ROOT / "models"
LLAMA_ROOT = ROOT / "llama"
STATE_DIR = RUN_ROOT / "state"
for _dir in (LOG_DIR, MODEL_DIR, LLAMA_ROOT, STATE_DIR):
    _dir.mkdir(parents=True, exist_ok=True)


def _pick_ts_dir() -> Path:
    override = os.environ.get("CLEF_TS_PREFIX")
    candidates = [Path(override)] if override else [Path("/opt/tailscale"), ROOT / "tailscale"]
    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
        except Exception:
            continue
        if os.access(candidate, os.W_OK):
            return candidate
    raise SystemExit("no writable directory for the tailscale binaries")


TS_DIR = _pick_ts_dir()
# Never the system daemon's socket or state: this runtime owns a private userspace tailscaled.
TS_SOCK = STATE_DIR / "tailscaled.sock"
TS_STATE = STATE_DIR / "tailscaled.state"


# ------------------------------------------------------------------- constants
ALIAS = "cloudflare/clef-flash"   # parent alias: documented here, never written by a notebook
PROVIDER = IDENT["hostname"]      # unique per runtime, e.g. clef-nb-<32 hex>
MODEL = "clef-flash"
SELF_TEST_MODEL = f"direct/{PROVIDER}/{MODEL}"
TS_API_BASE = "https://api.tailscale.com/api/v2"
TS_TAG = "tag:endpoints"
TS_AUTHKEY_EXPIRY_SECONDS = 3600
TS_HOSTNAME = IDENT["hostname"]   # unique per runtime, so no two notebooks share a node


# The bearer is generated in this kernel (see the identity block above) and handed to the
# registrar once. Nothing here reads or adopts another provider's api_key.
print(f"clef_remote_key: generated per run len={len(SECRETS['CLEF_REMOTE_KEY'])}")

GGUF_REPO = "bartowski/Cloudflare_clef-flash-GGUF"
GGUF_FILE = "Cloudflare_clef-flash-Q4_K_M.gguf"
GGUF_EXPECT_BYTES = 6_040_541_344
MMPROJ_FILE = "mmproj-Cloudflare_clef-flash-f16.gguf"
USE_MMPROJ = os.environ.get("CLEF_MMPROJ", "0").strip().lower() in ("1", "true", "yes", "on")

CTX_FULL = 262_144
CTX_T4 = 32_768
VRAM_HIGH_GIB = 22.0
VRAM_MIN_GIB = 14.0
KV_BYTES_PER_TOKEN = 32_768  # 8 full-attn layers * 4 kv heads * (256+256) * 2 B
CTX_OVERRIDE = os.environ.get("CLEF_CTX", "").strip()

# llama.cpp physical batch: a non-causal scorer must fit the whole prompt
# (state + question block) in one ubatch.  The old 512 default could not even
# hold the 2,008-token judge questions.  16384 matches the Mac rung and the
# 11,836-token judge state cap.  BATCH must be a multiple of UBATCH.
BATCH = int(os.environ.get("CLEF_BATCH", "16384"))
UBATCH = int(os.environ.get("CLEF_UBATCH", "16384"))
if BATCH % UBATCH:
    raise SystemExit(f"CLEF_BATCH={BATCH} is not a multiple of CLEF_UBATCH={UBATCH}")

WORKER_SPECS = ((0, 8091), (1, 8092))  # (CUDA_VISIBLE_DEVICES, loopback port)
SERVE_PORT_LADDER = (8080, 8443)
# The provider is created with timeoutMs=600000 (plexus_setup.py). The proxy gives up first, so
# its own 502 reaches Plexus before Plexus abandons the request. See README "Timeouts".
UPSTREAM_TIMEOUT = 540.0
READY_TIMEOUT = 900.0
SELF_TEST_TIMEOUT = 600.0
PROVIDER_TIMEOUT_MS = 600_000  # what the provider must allow; checked live at bring-up
MIN_PROVIDER_TIMEOUT_MS = 300_000  # below this a T4 pre-fill is cut off mid-request
PROBE_TIMEOUT = 20.0
WATCH_INTERVAL = 30.0
GPU_MIN_USED_MIB = 2048  # under this, a "GPU" load is really a CPU load with extra steps


def log(message: str) -> None:
    """Timestamped console line, mirrored into ROOT/logs/notebook.log."""
    print(f"{time.strftime('%H:%M:%S')} {message}", flush=True)
    try:
        with open(LOG_DIR / "notebook.log", "a") as handle:
            handle.write(f"{time.strftime('%Y-%m-%dT%H:%M:%S')} {message}\n")
    except Exception:
        pass


log(f"root={ROOT}  ts_dir={TS_DIR}  registrar={REGISTRY_URL}")
log(f"mmproj={'on' if USE_MMPROJ else 'off'}  ctx_override={CTX_OVERRIDE or '-'}")
log(f"batch={BATCH} ubatch={UBATCH}")
log(f"tailscaled state={TS_STATE} socket={TS_SOCK}")


In [ ]:
%%bash
# [shell 2/3] llama.cpp b12292 prebuilt CUDA release + its CUDA runtime libs.
# Built binaries only - no source build. Both tarballs extract flat into /content/llama.
set -euo pipefail
if [ -d /kaggle ]; then ROOT="${CLEF_ROOT:-/kaggle/temp/clef}"; else ROOT="${CLEF_ROOT:-/content}"; fi
[ -d "$ROOT" ] && [ -w "$ROOT" ] || ROOT="$HOME/.clef-colab"
mkdir -p "$ROOT/llama" "$ROOT/logs"
cd "$ROOT/llama"

BASE="https://github.com/ggml-org/llama.cpp/releases/download/b12292"
# Archive roots are not asset names: the binary asset unpacks into llama-b12292/, so the old
# `-d "$ASSET"` guard never matched and re-downloaded 198 MB on every run. Guard on the payload.
BIN_ASSET="llama-b12292-bin-ubuntu-cuda-12.8-x64"
BIN_DIR="llama-b12292"
CUDART_ASSET="cudart-llama-b12292-bin-ubuntu-cuda-12.8-x64"
CUDART_DIR="cudart-llama-b12292-bin-ubuntu-cuda-12.8-x64"

if [ -x "$BIN_DIR/llama-server" ] && [ -f "$BIN_DIR/libggml-cuda.so" ]; then
  echo "skip $BIN_ASSET (already extracted in $BIN_DIR)"
else
  echo "+ $BIN_ASSET.tar.gz"
  curl -fL --retry 3 --retry-delay 2 --progress-bar -o "$BIN_ASSET.tar.gz" "$BASE/$BIN_ASSET.tar.gz"
  tar -xzf "$BIN_ASSET.tar.gz"
  rm -f "$BIN_ASSET.tar.gz"
fi

if [ -f "$CUDART_DIR/libcudart.so.12" ]; then
  echo "skip $CUDART_ASSET (already extracted in $CUDART_DIR)"
else
  echo "+ $CUDART_ASSET.tar.gz"
  curl -fL --retry 3 --retry-delay 2 --progress-bar -o "$CUDART_ASSET.tar.gz" "$BASE/$CUDART_ASSET.tar.gz"
  tar -xzf "$CUDART_ASSET.tar.gz"
  rm -f "$CUDART_ASSET.tar.gz"
fi

chmod +x "$ROOT"/llama/llama-b*/llama-* 2>/dev/null || true
echo
echo "llama-server binaries:"
find "$ROOT/llama" -maxdepth 2 -name 'llama-server' -type f -print -exec ls -lh {} \;
echo
du -sh "$ROOT/llama"/* 2>/dev/null || true

In [ ]:
# [4/13] locate the binaries, prove /v1/systemone exists, fetch the GGUF
from __future__ import annotations

import os
import shutil
from pathlib import Path

# --- 4.1 llama-server -------------------------------------------------------
servers = [
    path
    for path in sorted(LLAMA_ROOT.rglob("llama-server"))
    if path.is_file() and os.access(path, os.X_OK)
]
if not servers:
    raise SystemExit(
        f"no executable llama-server under {LLAMA_ROOT} — run the llama.cpp release cell ([shell 2/3]) first"
    )
LLAMA_SERVER = servers[0]
CUDART_DIRS = [
    path for path in sorted(LLAMA_ROOT.iterdir()) if path.is_dir() and path.name.startswith("cudart")
]
LIB_PATH = os.pathsep.join([str(path) for path in CUDART_DIRS] + [str(LLAMA_SERVER.parent)])
log(f"llama-server : {LLAMA_SERVER}")
log(f"cudart dirs  : {[path.name for path in CUDART_DIRS] or 'none'}")


# --- 4.2 hard capability gate ----------------------------------------------
def _contains(path: Path, needle: bytes, chunk: int = 8 << 20) -> bool:
    """Streaming substring search over a large shared object."""
    carry = b""
    with open(path, "rb") as handle:
        while True:
            block = handle.read(chunk)
            if not block:
                return False
            haystack = carry + block
            if needle in haystack:
                return True
            carry = haystack[-len(needle) :]


impl_candidates = [path for path in sorted(LLAMA_ROOT.rglob("libllama-server-impl.so*")) if path.is_file()]
if not impl_candidates:
    raise SystemExit(f"libllama-server-impl.so not found under {LLAMA_ROOT}")
impl = next((path for path in impl_candidates if _contains(path, b"/v1/systemone")), None)
if impl is None:
    raise SystemExit(
        "ABORT: this llama.cpp build has no /v1/systemone. The `decisions` surface would be missing "
        "and every Plexus call would 404. Check that [shell 2/3] pulled release b12292."
    )
log(f"systemone    : OK in {impl.name}")
libllama = next((path for path in sorted(LLAMA_ROOT.rglob("libllama.so*")) if path.is_file()), None)
if libllama is not None:
    marker = "present" if _contains(libllama, b"clef") else "ABSENT"
    log(f"clef arch    : {marker} string in {libllama.name} (sanity only, not a gate)")

# --- 4.3 model --------------------------------------------------------------
try:
    from huggingface_hub import hf_hub_download
except ImportError as exc:  # pragma: no cover - only when cell 1 was skipped
    raise SystemExit("huggingface_hub is missing — run the pip cell ([shell 1/3])") from exc

log(f"downloading  : {GGUF_REPO}/{GGUF_FILE} (~6.04 GB on first run)")
GGUF = Path(hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE, cache_dir=str(MODEL_DIR)))
GGUF_BYTES = GGUF.stat().st_size
log(f"gguf         : {GGUF} ({GGUF_BYTES / 1e9:.2f} GB)")
if GGUF_BYTES != GGUF_EXPECT_BYTES:
    log(
        f"WARNING: gguf is {GGUF_BYTES} bytes, expected {GGUF_EXPECT_BYTES}; "
        "continuing (the HF file may have been re-uploaded)"
    )

MMPROJ = None
if USE_MMPROJ:
    MMPROJ = Path(hf_hub_download(repo_id=GGUF_REPO, filename=MMPROJ_FILE, cache_dir=str(MODEL_DIR)))
    log(f"mmproj       : {MMPROJ} ({MMPROJ.stat().st_size / 1e9:.2f} GB)")
else:
    log("mmproj       : off (CLEF_MMPROJ=1 to enable image input, costs VRAM)")

free_gb = shutil.disk_usage(ROOT).free / 1e9
log(f"disk free    : {free_gb:.1f} GB")
if free_gb < 2:
    log("WARNING: less than 2 GB free — llama-server stdout will fail once logs grow")

In [ ]:
# [5/13] GPU inventory, context policy, worker plan
from __future__ import annotations

import re
import shutil
import subprocess


def _nvidia_smi() -> list:
    exe = shutil.which("nvidia-smi") or "/usr/bin/nvidia-smi"
    proc = subprocess.run(
        [exe, "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if proc.returncode != 0:
        raise SystemExit(f"nvidia-smi failed rc={proc.returncode}: {proc.stderr.strip()[:400]}")
    found = []
    for line in proc.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 3:
            continue
        index = int(parts[0])
        name = ",".join(parts[1:-1])
        number, _, unit = parts[-1].partition(" ")
        gib = float(number) / 1024.0 if unit.upper().startswith("MI") else float(number)
        found.append({"index": index, "name": name, "gib": round(gib, 2), "raw": parts[-1]})
    return found


GPUS = _nvidia_smi()
if not GPUS:
    raise SystemExit(
        "no GPUs visible: Runtime → Change runtime type → T4 GPU (2× T4 recommended), then re-run"
    )
log(f"gpus         : {len(GPUS)}")
for gpu in GPUS:
    log(f"  cuda:{gpu['index']} {gpu['name']} {gpu['gib']:.2f} GiB")


def gpu_compute_memory() -> dict:
    """pid -> MiB held on a GPU, from nvidia-smi's compute-apps view.

    This is the evidence of record for "the weights are on the card": `-ngl 99` is a request,
    and neither a command line nor a log line proves it was honoured. Returns {} when the
    driver declines to enumerate compute processes.
    """
    exe = shutil.which("nvidia-smi") or "/usr/bin/nvidia-smi"
    try:
        proc = subprocess.run(
            [exe, "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=120,
        )
    except (OSError, subprocess.TimeoutExpired):
        return {}
    if proc.returncode != 0:
        return {}
    held = {}
    for line in proc.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        try:
            held[int(parts[0])] = int(float(parts[1]))
        except ValueError:
            continue
    return held


_OFFLOAD = re.compile(r"offload\w*[^\n]*?(\d+)\s*/\s*(\d+)\s+layers?", re.IGNORECASE)


def gpu_layers_in_log(path) -> tuple | None:
    """(layers_on_gpu, total_layers) from llama.cpp's load log, or None if absent.

    Deliberately loose: the wording drifts between releases ("offloaded 33/33 layers to GPU",
    "offloading 32 repeating layers"). Only consulted when nvidia-smi cannot enumerate compute
    processes, so a missing line means "unproven", never "assume GPU".
    """
    try:
        with open(path, "r", errors="replace") as handle:
            text = handle.read()
    except OSError:
        return None
    matches = _OFFLOAD.findall(text)
    if not matches:
        return None
    on_gpu, total = matches[-1]
    return int(on_gpu), int(total)


def ctx_for(gib: float) -> int:
    """Context for one GPU, from that GPU's total VRAM."""
    if CTX_OVERRIDE:
        try:
            wanted = int(CTX_OVERRIDE)
        except ValueError:
            raise SystemExit(f"CLEF_CTX={CTX_OVERRIDE!r} is not an integer") from None
    elif gib >= VRAM_HIGH_GIB:
        wanted = CTX_FULL
    elif gib >= VRAM_MIN_GIB:
        wanted = CTX_T4
    else:
        raise SystemExit(
            f"{gib:.2f} GiB VRAM is below the {VRAM_MIN_GIB:.0f} GiB floor: the 6.04 GB model leaves "
            "no room for a usable KV cache. Use a T4 or larger runtime."
        )
    return min(wanted, CTX_FULL)


# GPU policy: 1 GPU -> one worker pinned to cuda:0; 2+ GPUs -> one worker per GPU (0 and 1).
# Never one worker spanning both GPUs, never a silent 2 -> 1 degrade.
if len(GPUS) == 1:
    log(
        "gpus         : exactly one physical GPU attached — allowing ONE worker pinned to cuda:0 "
        "(permitted only because there is a single device)"
    )
    assigned = GPUS[:1]
else:
    assigned = GPUS[:2]
    if len(GPUS) > 2:
        log(f"gpus         : ignoring extra GPUs {[gpu['index'] for gpu in GPUS[2:]]} — using 0 and 1")

WORKER_PLAN = []
for slot, gpu in enumerate(assigned):
    cuda_device, port = WORKER_SPECS[slot]
    context = ctx_for(gpu["gib"])
    WORKER_PLAN.append(
        {
            "slot": slot,
            "cuda": cuda_device,
            "port": port,
            "ctx": context,
            "kv_gib": context * KV_BYTES_PER_TOKEN / 2**30,
        }
    )
    log(
        f"worker {slot}     : cuda:{cuda_device} port={port} ctx={context} "
        f"kv={WORKER_PLAN[-1]['kv_gib']:.2f} GiB"
    )

if len(WORKER_PLAN) != len(assigned):
    raise SystemExit("worker plan does not cover every assigned GPU")
if len(GPUS) >= 2 and len(WORKER_PLAN) != 2:
    raise SystemExit("two physical GPUs must produce two workers — refusing to degrade silently")
if any(worker["ctx"] > CTX_FULL for worker in WORKER_PLAN):
    raise SystemExit(f"context above the {CTX_FULL} cap")

In [ ]:
%%bash
# [shell 3/3] tailscale static binaries (userspace networking: no /dev/net/tun, no systemd here).
# Do NOT use install.sh on Colab - it wants systemd.
set -euo pipefail
if [ -d /kaggle ]; then ROOT="${CLEF_ROOT:-/kaggle/temp/clef}"; else ROOT="${CLEF_ROOT:-/content}"; fi
[ -d "$ROOT" ] && [ -w "$ROOT" ] || ROOT="$HOME/.clef-colab"
mkdir -p "$ROOT/logs"

TS_DIR="${CLEF_TS_PREFIX:-/opt/tailscale}"
if ! mkdir -p "$TS_DIR" 2>/dev/null; then TS_DIR="$ROOT/tailscale"; mkdir -p "$TS_DIR"; fi
if [ ! -w "$TS_DIR" ]; then TS_DIR="$ROOT/tailscale"; mkdir -p "$TS_DIR"; fi

if [ -x "$TS_DIR/tailscaled" ] && [ -x "$TS_DIR/tailscale" ]; then
  echo "already installed in $TS_DIR"
else
  TMP="$(mktemp -d)"
  curl -fL --retry 3 --retry-delay 2 --progress-bar \
    -o "$TMP/ts.tgz" https://pkgs.tailscale.com/stable/tailscale_latest_amd64.tgz
  tar -xzf "$TMP/ts.tgz" -C "$TMP"
  install -m 0755 "$TMP"/tailscale_*/tailscale  "$TS_DIR/tailscale"
  install -m 0755 "$TMP"/tailscale_*/tailscaled "$TS_DIR/tailscaled"
  rm -rf "$TMP"
fi

mkdir -p /var/lib/tailscale 2>/dev/null || true
"$TS_DIR/tailscale" version
echo "TS_DIR=$TS_DIR"

In [ ]:
# [7/13] Plexus management client — alias + provider, every write read back
from __future__ import annotations

import json
import urllib.error
import urllib.request




class PlexusClient:
    """Read-only description of this runtime's routing state.

    v10 notebooks hold no Plexus credential and never write a provider or an alias: the
    registrar owns both, so the inherited management client is replaced rather than left as
    dead code that a later edit could re-arm.
    """

    def __init__(self, provider: str = PROVIDER) -> None:
        self.provider = provider

    def describe_rung(self) -> str:
        if globals().get("RUNTIME") is not None and getattr(RUNTIME, "lease", None) is not None:
            return f"notebook pool clef-flash-notebooks -> {PROVIDER} (owned by the registrar)"
        return "not published"


In [ ]:
# [8/13] llama-server workers + the round-robin proxy that fronts them
from __future__ import annotations

import hmac
import http.client
import json
import os
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

BEARER = SECRETS["CLEF_REMOTE_KEY"]


def http_json(port: int, path: str, timeout: float = 5.0, bearer: str | None = None) -> dict:
    request = urllib.request.Request(f"http://127.0.0.1:{port}{path}")
    if bearer:
        request.add_header("authorization", f"Bearer {bearer}")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        raw = response.read()
        return json.loads(raw) if raw else {}


class Worker:
    """One llama-server process pinned to one physical GPU, loopback only."""

    def __init__(
        self,
        slot: int,
        cuda: int,
        port: int,
        ctx: int,
        model_path,
        mmproj=None,
        bearer: str = BEARER,
    ) -> None:
        self.slot = slot
        self.cuda = cuda
        self.port = port
        self.ctx = ctx
        self.model = model_path
        self.mmproj = mmproj
        self.bearer = bearer
        self.log_path = LOG_DIR / f"llama-{slot}.log"
        self.proc = None
        self._logfh = None

    def command(self) -> list:
        cmd = [
            str(LLAMA_SERVER),
            "-m",
            str(self.model),
            "--host",
            "127.0.0.1",
            "--port",
            str(self.port),
            "-c",
            str(self.ctx),
            "-b",
            str(BATCH),
            "-ub",
            str(UBATCH),
            "-np",
            "1",
            "-ngl",
            "99",
            "--api-key",
            self.bearer,
        ]
        if self.mmproj:
            cmd += ["--mmproj", str(self.mmproj)]
        return cmd

    def start(self) -> None:
        env = dict(os.environ)
        env["CUDA_VISIBLE_DEVICES"] = str(self.cuda)
        env["LD_LIBRARY_PATH"] = LIB_PATH + (
            os.pathsep + env["LD_LIBRARY_PATH"] if env.get("LD_LIBRARY_PATH") else ""
        )
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self._logfh = open(self.log_path, "ab", buffering=0)
        cmd = self.command()
        printable = " ".join("***" if arg == self.bearer else arg for arg in cmd)
        log(f"worker {self.slot}     : CUDA_VISIBLE_DEVICES={self.cuda} {printable}")
        self.proc = subprocess.Popen(
            cmd,
            stdout=self._logfh,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )
        log(f"worker {self.slot}     : pid={self.proc.pid} log={self.log_path}")

    def alive(self) -> bool:
        return self.proc is not None and self.proc.poll() is None

    def healthy(self, timeout: float = 3.0) -> bool:
        if not self.alive():
            return False
        try:
            body = http_json(self.port, "/health", timeout=timeout, bearer=self.bearer)
        except Exception:
            return False
        return str(body.get("status", "")).lower() == "ok"

    def props(self) -> dict:
        return http_json(self.port, "/props", timeout=30.0, bearer=self.bearer)

    def wait_ready(self, timeout: float = READY_TIMEOUT) -> dict:
        deadline = time.time() + timeout
        while time.time() < deadline:
            if self.proc is not None and self.proc.poll() is not None:
                raise RuntimeError(
                    f"worker {self.slot} exited rc={self.proc.returncode} while loading "
                    f"(see {self.log_path})\n{self.tail()}"
                )
            if self.healthy():
                return self.props()
            time.sleep(2.0)
        raise RuntimeError(f"worker {self.slot} never became healthy in {timeout:.0f}s\n{self.tail()}")

    def tail(self, lines: int = 40) -> str:
        try:
            with open(self.log_path, "r", errors="replace") as handle:
                return "".join(handle.readlines()[-lines:])
        except FileNotFoundError:
            return "(no log)"

    def stop(self, grace: float = 15.0) -> None:
        proc = self.proc
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    proc.terminate()
                except Exception:
                    pass
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                log(f"worker {self.slot}     : ignored SIGTERM for {grace:.0f}s — SIGKILL")
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except (ProcessLookupError, PermissionError, OSError):
                    try:
                        proc.kill()
                    except Exception:
                        pass
                try:
                    proc.wait(timeout=10)
                except Exception:
                    pass
        self.proc = None
        if self._logfh is not None:
            try:
                self._logfh.close()
            except Exception:
                pass
            self._logfh = None


class RoundRobinProxy:
    """Loopback front door for the workers.

    `Authorization: Bearer <CLEF_REMOTE_KEY>` is required on every inference route (401
    otherwise). POST /v1/systemone is spread round-robin across live workers and retried on the
    next worker after a connection failure. GET /health is deliberately unauthenticated: it is
    liveness only, and it is what the operator polls from the tailnet.
    """

    def __init__(self, workers, port: int, bearer: str = BEARER, upstream_timeout: float = UPSTREAM_TIMEOUT) -> None:
        self.workers = list(workers)
        self.port = port
        self.bearer = bearer
        self.upstream_timeout = upstream_timeout
        self.requests = 0
        self.served = 0
        self.failures = 0
        # In-flight authorized inference requests: teardown drains these before stopping workers.
        self.active = 0
        self._rr = 0
        self._lock = threading.Lock()
        self._server = None
        self._thread = None

    # ---------------------------------------------------------------- lifecycle
    def start(self) -> None:
        proxy = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = "clef-rr-proxy"

            def log_message(self, fmt: str, *args) -> None:  # noqa: A003 - stdlib signature
                log(f"proxy        : {self.address_string()} {fmt % args}")

            # --- helpers
            def _send(self, code: int, body: bytes, ctype: str = "application/json") -> None:
                self.send_response(code)
                self.send_header("content-type", ctype)
                self.send_header("content-length", str(len(body)))
                self.send_header("connection", "close")
                self.end_headers()
                self.close_connection = True
                try:
                    self.wfile.write(body)
                except (BrokenPipeError, ConnectionResetError):
                    pass

            def _json(self, code: int, payload: dict) -> None:
                self._send(code, json.dumps(payload).encode())

            def _unauthorized(self) -> None:
                self._json(401, {"error": "unauthorized", "detail": "Authorization: Bearer <key> required"})

            def _authorized(self) -> bool:
                header = self.headers.get("authorization", "") or ""
                if not header.lower().startswith("bearer "):
                    return False
                return hmac.compare_digest(header[7:].strip(), proxy.bearer)

            def _read_body(self) -> bytes | None:
                length = self.headers.get("content-length")
                if length is None:
                    if (self.headers.get("transfer-encoding") or "").lower() == "chunked":
                        self._json(411, {"error": "chunked bodies are not supported; send content-length"})
                        return None
                    return b""
                try:
                    return self.rfile.read(int(length))
                except (ValueError, OSError):
                    self._json(400, {"error": "unreadable content-length"})
                    return None

            def _forward(self, method: str, path: str, body: bytes | None) -> None:
                headers = {
                    "authorization": f"Bearer {proxy.bearer}",
                    "content-type": "application/json",
                    "accept": "application/json",
                    "connection": "close",
                }
                last_error = None
                for worker in proxy.order():
                    try:
                        conn = http.client.HTTPConnection(
                            "127.0.0.1", worker.port, timeout=proxy.upstream_timeout
                        )
                        conn.request(method, path, body=body, headers=headers)
                        response = conn.getresponse()
                        data = response.read()
                        status = response.status
                        ctype = response.getheader("content-type") or "application/json"
                        conn.close()
                        with proxy._lock:
                            proxy.served += 1
                        self._send(status, data, ctype)
                        return
                    except socket.timeout as exc:
                        last_error = exc
                        with proxy._lock:
                            proxy.failures += 1
                        log(f"proxy        : worker {worker.slot}:{worker.port} timed out; trying next")
                    except (ConnectionError, OSError, http.client.HTTPException) as exc:
                        last_error = exc
                        with proxy._lock:
                            proxy.failures += 1
                        log(f"proxy        : worker {worker.slot}:{worker.port} failed ({exc!r}); trying next")
                self._json(502, {"error": "no worker answered", "detail": repr(last_error)})

            # --- routes
            def do_GET(self) -> None:  # noqa: N802 - stdlib signature
                path = self.path.split("?", 1)[0]
                if path == "/health":
                    self._json(
                        200,
                        {
                            "status": "ok",
                            # The registrar proves it is talking to THIS runtime, not a stale
                            # node that reused the name.
                            "run_id": RUN_ID,
                            "workers": [
                                {
                                    "slot": worker.slot,
                                    "port": worker.port,
                                    "cuda": worker.cuda,
                                    "alive": worker.alive(),
                                    "healthy": worker.healthy(),
                                }
                                for worker in proxy.workers
                            ],
                            "requests": proxy.requests,
                            "served": proxy.served,
                            "failures": proxy.failures,
                            "active": proxy.active,
                        },
                    )
                    return
                if path in ("/v1/models", "/models"):
                    if not self._authorized():
                        self._unauthorized()
                        return
                    with proxy._lock:
                        proxy.requests += 1
                    self._forward("GET", "/v1/models", None)
                    return
                self._json(404, {"error": "not found", "path": path})

            def do_POST(self) -> None:  # noqa: N802 - stdlib signature
                path = self.path.split("?", 1)[0]
                if path not in ("/v1/systemone", "/systemone"):
                    self._json(404, {"error": "not found", "path": path, "hint": "POST /v1/systemone"})
                    return
                if not self._authorized():
                    self._unauthorized()
                    return
                body = self._read_body()
                if body is None:
                    return
                with proxy._lock:
                    proxy.requests += 1
                    proxy.active += 1
                try:
                    self._forward("POST", "/v1/systemone", body)
                finally:
                    with proxy._lock:
                        proxy.active -= 1

        self._server = ThreadingHTTPServer(("127.0.0.1", self.port), Handler)
        self._server.daemon_threads = True
        self._thread = threading.Thread(target=self._server.serve_forever, name="rr-proxy", daemon=True)
        self._thread.start()

    def order(self) -> list:
        """Live workers, round-robin starting point; falls back to every worker if none are up."""
        with self._lock:
            live = [worker for worker in self.workers if worker.alive()] or list(self.workers)
            start = self._rr % len(live)
            self._rr += 1
            return live[start:] + live[:start]

    def stop(self) -> None:
        server, self._server = self._server, None
        if server is None:
            return
        try:
            server.shutdown()
        except Exception:
            pass
        try:
            server.server_close()
        except Exception:
            pass
        if self._thread is not None:
            try:
                self._thread.join(timeout=5)
            except Exception:
                pass
            self._thread = None


In [ ]:
# [9/13] tailscale: userspace daemon, ephemeral join, TCP serve
from __future__ import annotations

import json
import os
import re
import signal
import socket
import subprocess
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

NODE_STALE_SECONDS = 300


class Tailscale:
    """Userspace-mode tailscaled + CLI, on a private state file and socket."""

    def __init__(
        self,
        ts_dir=TS_DIR,
        sock=TS_SOCK,
        state=TS_STATE,
        hostname: str = TS_HOSTNAME,
        api_key: str = "",
    ) -> None:
        self.dir = Path(ts_dir)
        self.sock = Path(sock)
        self.state = Path(state)
        self.hostname = hostname
        self.api_key = api_key or SECRETS["TAILSCALE_API_KEY"]
        self.proc = None
        self._logfh = None
        self.device_id = None
        self.auth_key_id = None

    @property
    def cli(self) -> str:
        return str(self.dir / "tailscale")

    @property
    def daemon(self) -> str:
        return str(self.dir / "tailscaled")

    @staticmethod
    def _redacted(args: list) -> str:
        """Command line safe to log: an authkey never survives into a message."""
        return " ".join(
            "--authkey=***" if str(arg).startswith("--authkey=") else str(arg) for arg in args
        )

    def _run(self, args: list, timeout: float = 60.0, check: bool = True) -> subprocess.CompletedProcess:
        try:
            proc = subprocess.run(
                [self.cli, f"--socket={self.sock}", *args],
                capture_output=True,
                text=True,
                timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            # TimeoutExpired.__str__ embeds the whole argv, i.e. --authkey=<secret>
            raise RuntimeError(
                f"tailscale {self._redacted(args)} timed out after {timeout:.0f}s"
            ) from None
        if check and proc.returncode != 0:
            detail = (proc.stderr.strip() or proc.stdout.strip())[:600]
            raise RuntimeError(f"tailscale {self._redacted(args)} rc={proc.returncode}: {detail}")
        return proc

    # ------------------------------------------------------------- tailscale api
    def _api(self, method: str, path: str, payload: dict | None = None, timeout: float = 45.0, allowed: tuple = (200,)) -> tuple:
        """Call the Tailscale REST API. Never logs the API token or an auth key value."""
        data = json.dumps(payload).encode() if payload is not None else None
        request = urllib.request.Request(f"{TS_API_BASE}{path}", data=data, method=method)
        request.add_header("authorization", f"Bearer {self.api_key}")
        request.add_header("accept", "application/json")
        if data is not None:
            request.add_header("content-type", "application/json")
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                status = response.status
                raw = response.read()
        except urllib.error.HTTPError as exc:
            body = exc.read()[:600].decode("utf-8", "replace")
            raise RuntimeError(f"tailscale api {method} {path} -> {exc.code}: {body}") from None
        except urllib.error.URLError as exc:
            raise RuntimeError(f"tailscale api {method} {path} -> unreachable: {exc.reason}") from None
        if status not in allowed:
            raise RuntimeError(f"tailscale api {method} {path} -> unexpected status {status}")
        if not raw:
            return status, {}
        try:
            return status, json.loads(raw)
        except json.JSONDecodeError:
            return status, {}

    def devices(self) -> list:
        _status, body = self._api("GET", "/tailnet/-/devices")
        devices = body.get("devices") if isinstance(body, dict) else None
        if not isinstance(devices, list):
            raise RuntimeError(f"Tailscale device list had an unexpected shape: {body!r:.400}")
        return devices





    def mint_auth_key(self, expiry_seconds: int = TS_AUTHKEY_EXPIRY_SECONDS) -> str:
        """Mint a single-use, ephemeral, preauthorised key for this join, then return its value."""
        description = f"clef-llama-remote-colab-v2-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}"
        _status, body = self._api(
            "POST",
            "/tailnet/-/keys",
            {
                "capabilities": {"devices": {"create": {"reusable": False, "ephemeral": True, "preauthorized": True, "tags": [TS_TAG]}}},
                "expirySeconds": int(expiry_seconds),
                "description": description,
            },
        )
        key = body.get("key") if isinstance(body, dict) else ""
        self.auth_key_id = body.get("id") if isinstance(body, dict) else None
        if not isinstance(key, str) or not key.startswith("tskey-auth-"):
            raise RuntimeError("Tailscale did not return an auth key value for this join")
        log(f"tailscale    : minted single-use ephemeral auth key (id={self.auth_key_id})")
        return key

    def revoke_auth_key(self) -> bool:
        """Best-effort delete of the auth key minted for this run."""
        key_id, self.auth_key_id = self.auth_key_id, None
        if not key_id:
            return False
        try:
            self._api("DELETE", f"/tailnet/-/keys/{key_id}", allowed=(200, 204, 404))
            log("tailscale    : per-run auth key revoked")
            return True
        except Exception as exc:
            log(f"tailscale    : auth-key revoke failed ({exc})")
            return False

    def own_device_id(self) -> str | None:
        """This run's Tailscale device id, matched by node id or tailnet addresses."""
        state = self.status()
        self_node = state.get("Self") or {}
        node_id = self_node.get("ID")
        addresses = set(self_node.get("TailscaleIPs") or [])
        for device in self.devices():
            if node_id and device.get("nodeId") == node_id:
                return str(device.get("id") or "") or None
            if addresses and addresses & set(device.get("addresses") or []):
                return str(device.get("id") or "") or None
        return None

    def delete_device(self, device_id):
        """Delete this run's own Tailscale device, and nothing else.

        Ownership is resolved from live status (Self.ID / tailnet addresses) at deletion time. A
        hostname match is not ownership: another runtime may legitimately hold a different device,
        and deleting it would take a peer's route away.
        """
        if not device_id:
            return False
        current = self.own_device_id()
        if current != device_id:
            raise RuntimeError("refusing to delete a Tailscale device not owned by this runtime")
        self._api("DELETE", f"/device/{device_id}", allowed=(200, 204, 404))
        if any(str(d.get("id")) == str(device_id) for d in self.devices()):
            raise RuntimeError("own Tailscale device deletion not confirmed")
        return True

    # ------------------------------------------------------------------ daemon
    def start_daemon(self, timeout: float = 90.0) -> None:
        for path in (self.daemon, self.cli):
            if not Path(path).exists():
                raise SystemExit(f"{path} missing — run the tailscale cell ([shell 3/3]) first")
        if self.proc is not None and self.proc.poll() is None:
            log("tailscaled   : already running")
            return
        self.sock.parent.mkdir(parents=True, exist_ok=True)
        self.state.parent.mkdir(parents=True, exist_ok=True)
        log_path = LOG_DIR / "tailscaled.log"
        self._logfh = open(log_path, "ab", buffering=0)
        cmd = [
            self.daemon,
            "--tun=userspace-networking",
            "--socks5-server=localhost:1055",
            "--outbound-http-proxy-listen=localhost:1055",
            f"--state={self.state}",
            f"--socket={self.sock}",
        ]
        log(f"tailscaled   : {' '.join(cmd)}")
        self.proc = subprocess.Popen(
            cmd, stdout=self._logfh, stderr=subprocess.STDOUT, start_new_session=True
        )
        deadline = time.time() + timeout
        while time.time() < deadline:
            if self.proc.poll() is not None:
                raise RuntimeError(f"tailscaled exited rc={self.proc.returncode}; see {log_path}")
            if self._run(["status", "--json"], timeout=20.0, check=False).stdout.strip().startswith("{"):
                log(f"tailscaled   : socket answering ({self.sock})")
                return
            time.sleep(1.0)
        raise RuntimeError(f"tailscaled socket did not answer within {timeout:.0f}s; see {log_path}")

    # -------------------------------------------------------------------- state
    def status(self) -> dict:
        proc = self._run(["status", "--json"], timeout=30.0, check=False)
        text = proc.stdout.strip()
        if not text.startswith("{"):
            return {}
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {}

    def backend_state(self) -> str:
        return str(self.status().get("BackendState", "unknown"))

    # --------------------------------------------------------------------- join
    def up(self, authkey: str, timeout: float = 180.0) -> dict:
        proc = self._run(
            [
                "up",
                f"--authkey={authkey}",
                f"--hostname={self.hostname}",
                "--accept-dns=false",
                "--ssh=false",
            ],
            timeout=timeout,
            check=False,
        )
        if proc.returncode != 0:
            detail = (proc.stderr.strip() or proc.stdout.strip())[:600]
            raise RuntimeError(f"tailscale up failed rc={proc.returncode}: {detail}")
        return self.wait_running()

    def wait_running(self, timeout: float = 120.0) -> dict:
        deadline = time.time() + timeout
        state = {}
        while time.time() < deadline:
            state = self.status()
            if str(state.get("BackendState")) == "Running":
                return state
            time.sleep(2.0)
        raise RuntimeError(
            f"tailnet never reached Running (BackendState={state.get('BackendState')!r}); "
            "an expired or already-consumed authkey is the usual cause"
        )

    def ipv4(self) -> str:
        proc = self._run(["ip", "-4"], timeout=30.0, check=False)
        lines = [line.strip() for line in proc.stdout.splitlines() if line.strip()]
        ip = lines[0] if lines else ""
        if not ip.startswith("100."):
            raise RuntimeError(f"expected a 100.x tailnet IPv4, got {ip!r}")
        return ip

    def dnsname(self) -> str:
        return str((self.status().get("Self") or {}).get("DNSName", "")).rstrip(".")

    # -------------------------------------------------------------------- serve
    def serve_tcp(self, port: int) -> None:
        """Publish tcp/<port> on the tailnet, forwarding to the same loopback port."""
        self._run(["serve", "--bg", f"--tcp={port}", f"tcp://127.0.0.1:{port}"], timeout=90.0)

    def serve_reset(self) -> None:
        self._run(["serve", "reset"], timeout=60.0, check=False)

    # -------------------------------------------------------------------- probe
    def probe(self, ip: str, port: int, path: str = "/health", timeout: float = PROBE_TIMEOUT) -> tuple:
        """GET http://<ip>:<port><path> through tailscaled's userspace SOCKS5 proxy, from here.

        Colab has no /dev/net/tun, so a plain socket to 100.x cannot leave this netns; the
        daemon's outbound proxy is the only local route onto the tailnet, and a request sent
        through it leaves as a tailnet packet and comes back in through `tailscale serve`. That
        makes it a check that the published port answers, not merely that serve is configured.
        Returns (ok, detail); never raises.
        """

        def read_exact(sock, count: int) -> bytes:
            buf = b""
            while len(buf) < count:
                block = sock.recv(count - len(buf))
                if not block:
                    raise OSError("SOCKS5 proxy closed mid-handshake")
                buf += block
            return buf

        try:
            sock = socket.create_connection(("127.0.0.1", 1055), timeout=timeout)
        except OSError as exc:
            return False, f"tailscaled outbound proxy 127.0.0.1:1055 unreachable ({exc})"
        try:
            sock.settimeout(timeout)
            sock.sendall(b"\x05\x01\x00")
            if read_exact(sock, 2) != b"\x05\x00":
                return False, "SOCKS5 proxy rejected the no-auth greeting"
            host = ip.encode()
            sock.sendall(b"\x05\x01\x00\x03" + bytes([len(host)]) + host + port.to_bytes(2, "big"))
            head = read_exact(sock, 4)
            if head[1] != 0:
                return False, f"SOCKS5 CONNECT {ip}:{port} refused (rep={head[1]})"
            if head[3] == 1:
                read_exact(sock, 6)
            elif head[3] == 4:
                read_exact(sock, 18)
            elif head[3] == 3:
                read_exact(sock, read_exact(sock, 1)[0] + 2)
            else:
                return False, f"SOCKS5 CONNECT {ip}:{port} returned atyp={head[3]}"
            sock.sendall(
                f"GET {path} HTTP/1.0\r\nhost: {ip}:{port}\r\nconnection: close\r\n\r\n".encode()
            )
            chunks = []
            while True:
                block = sock.recv(4096)
                if not block:
                    break
                chunks.append(block)
            raw = b"".join(chunks)
            status = raw.split(b"\r\n", 1)[0].decode("latin-1", "replace").strip()
            if " 200" not in status:
                return False, f"{ip}:{port}{path} -> {status or 'empty response'}"
            return True, f"{ip}:{port}{path} -> {status}"
        except (OSError, ValueError) as exc:
            return False, f"{ip}:{port}{path} failed: {exc}"
        finally:
            try:
                sock.close()
            except Exception:
                pass

    def down(self) -> None:
        self._run(["down"], timeout=90.0, check=False)

    def stop_daemon(self, grace: float = 10.0) -> None:
        proc, self.proc = self.proc, None
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    proc.terminate()
                except Exception:
                    pass
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except (ProcessLookupError, PermissionError, OSError):
                    try:
                        proc.kill()
                    except Exception:
                        pass
        if self._logfh is not None:
            try:
                self._logfh.close()
            except Exception:
                pass
            self._logfh = None


In [ ]:
# [10/13] Runtime: bring-up order, teardown order, heartbeat
from __future__ import annotations

import atexit
import json
import os
import signal
import threading
import time
import urllib.error
import urllib.request


def _opener():
    """urllib opener that honours CLEF_TAILNET_PROXY.

    With --tun=userspace-networking the container has no route to 100.x: a direct urllib
    request to a tailnet name never completes, so every tailnet request must ride
    tailscaled's local proxy. Before the daemon is up the env var is unset and this is a
    plain opener, so public calls (api.tailscale.com) keep going out directly.
    """
    proxy = os.environ.get("CLEF_TAILNET_PROXY")
    if proxy:
        return urllib.request.build_opener(urllib.request.ProxyHandler({"http": proxy, "https": proxy}))
    return urllib.request.build_opener()


class SelfTestFailure(RuntimeError):
    pass


class Runtime:
    """Owns the whole ephemeral node: workers, proxy, tailnet, Plexus registration.

    Bring-up:  workers -> GPU residency proof -> tailscale -> proxy -> tailnet reachability
               -> provider base URL/concurrency/timeout -> rung enabled.
    Teardown:  rung disabled FIRST (so Plexus stops routing), then the configured concurrency,
               then workers, then the proxy, then the tailnet.
    """

    def __init__(self, client: PlexusClient | None = None, ts: Tailscale | None = None) -> None:
        self.client = client or PlexusClient()
        self.ts = ts or Tailscale()
        self.workers: list = []
        self.proxy = None
        self.serve_port = None
        self.ip = None
        self.dnsname = None
        self.device_id = None
        self.base_url = None
        self.lease = None
        self._vm_lock = None
        self._stop = threading.Event()
        self._done = threading.Event()
        self._guard = threading.Lock()
        self._stopped = False
        self._rung_on = False
        self._armed = False
        self._heartbeat = None
        self.gpu_mib = {}
        self.concurrency = None
        self.provider_timeout_ms = None

    # ------------------------------------------------------------------ bring-up
    def start(self) -> "Runtime":
        with self._guard:
            if self._stopped:
                raise RuntimeError("this Runtime was already stopped — construct a new one")
            if self.workers or self.proxy is not None:
                raise RuntimeError("Runtime.start() was already called in this kernel")
            if self._vm_lock is not None:
                raise RuntimeError("bring-up already started in this kernel")
        # One runtime per VM. A second kernel would contend for the GPUs, for the loopback worker
        # ports and for tailscaled's single userspace proxy port, so it is refused before anything
        # is spawned. Separate VMs remain fully independent.
        lock_path = ROOT / "clef-runtime.lock"
        lock_path.parent.mkdir(parents=True, exist_ok=True)
        lock_handle = open(lock_path, "a+")
        try:
            fcntl.flock(lock_handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError:
            lock_handle.close()
            raise RuntimeError(
                f"another kernel on this VM holds {lock_path}: refusing to share the GPUs, the "
                "worker ports or the userspace tailscale proxy. Stop that runtime first; separate "
                "Colab/Kaggle runtimes are independent and supported."
            ) from None
        with self._guard:
            if self._stopped:
                fcntl.flock(lock_handle.fileno(), fcntl.LOCK_UN)
                lock_handle.close()
                raise RuntimeError("teardown requested before bring-up started")
            self._vm_lock = lock_handle
        self._arm_callbacks()
        log(
            "bring-up     : workers -> gpu proof -> tailscale -> proxy -> reachability "
            "-> registrar -> admission"
        )
        try:
            for stage in (
                self._start_workers,
                self._prove_gpu,
                self._start_tailnet,
                self._start_proxy,
                self._prove_reachable,
                self._prove_plexus,
                self._register_provider,
            ):
                self._mark(stage.__name__)
                if stage in (self._start_tailnet, self._start_proxy, self._register_provider):
                    # These phases create external state (tailnet, Serve, lease). Serialize them
                    # with stop() so teardown cannot miss a late side effect.
                    with self._guard:
                        self._ensure_running()
                        stage()
                else:
                    self._ensure_running()
                    stage()
            # Admission is the ONLY enable operation: the registrar creates this run's provider,
            # proves it through Plexus and enables that single pool target. It is serialized with
            # stop() so a teardown that won the guard can never be reversed by a late admission.
            self._ensure_running()
            self._mark("admission")
            with self._guard:
                self._ensure_running()
                self._self_test()
            self._ensure_running()
        except BaseException:
            log("bring-up     : failed — tearing down (lease withdrawn first)")
            self.stop(reason="bring-up failed")
            raise
        self._heartbeat = threading.Thread(target=self._watch, name="heartbeat", daemon=True)
        self._heartbeat.start()
        self.status()
        self._mark("ready")
        log("bring-up     : ready")
        return self

    def _ensure_running(self) -> None:
        if self._stop.is_set() or self._stopped:
            raise RuntimeError("teardown requested during bring-up — refusing to continue")

    def _mark(self, stage: str) -> None:
        """Breadcrumb written BEFORE each bring-up stage so a hard kill leaves a trail.

        Kaggle reports a killed session as CANCEL_ACKNOWLEDGED with no traceback; the last
        stage on disk is then the only evidence of how far the run actually got.
        """
        log(f"stage        : {stage}")
        base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
        # A hard kill uploads nothing we did not already write: keep the worker logs there too.
        try:
            import glob
            for src in glob.glob(str(LOG_DIR / "llama-*.log")):
                with open(src, "r", errors="replace") as fh:
                    tail = fh.readlines()[-250:]
                with open(os.path.join(base, os.path.basename(src)), "w") as out:
                    out.writelines(tail)
        except Exception:
            pass
        try:
            path = os.path.join(base, "BRINGUP_STAGE.json")
            with open(path, "w") as fh:
                fh.write(json.dumps({"stage": stage, "t": time.time(),
                                     "iso": time.strftime("%Y-%m-%dT%H:%M:%S")}) + "\n")
                fh.flush()
                os.fsync(fh.fileno())
        except Exception:
            pass

    def _start_workers(self) -> None:
        for spec in WORKER_PLAN:
            worker = Worker(
                slot=spec["slot"],
                cuda=spec["cuda"],
                port=spec["port"],
                ctx=spec["ctx"],
                model_path=GGUF,
                mmproj=MMPROJ,
            )
            # Spawn inside the guard: a teardown arriving mid-bring-up must never sweep the worker
            # list between append() and Popen(), which would leave a llama-server holding ~10 GB of
            # VRAM on a node whose rung is already disabled.
            with self._guard:
                if self._stopped:
                    raise RuntimeError("teardown requested during bring-up — not starting another worker")
                worker.start()
                self.workers.append(worker)
        for worker in self.workers:
            props = worker.wait_ready()
            settings = props.get("default_generation_settings") or {}
            reported = settings.get("n_ctx")
            log(f"worker {worker.slot}     : ready n_ctx={reported} (requested {worker.ctx})")
            if reported is not None and int(reported) != worker.ctx:
                log(
                    f"WARNING: worker {worker.slot} reports n_ctx={reported} but {worker.ctx} was "
                    "requested — the server clamped it (VRAM or --parallel)"
                )

    def _prove_gpu(self) -> None:
        """Refuse to continue unless every worker is provably holding GPU memory.

        `-ngl 99` is a request, not a result: a CPU build, a stale binary or a CUDA init that
        fell back all keep the weights on the host and still answer /health. Enabling the rung on
        such a node burns the provider timeout on every call, so this gate is what keeps it off.

        Evidence order: nvidia-smi compute-apps memory per worker pid (primary, no log parsing),
        then an "offloaded N/M layers" line from the worker log (secondary, only when the driver
        declines to enumerate compute processes).
        """
        deadline = time.time() + 30.0
        held = {}
        pending = []
        while True:
            held = gpu_compute_memory()
            pending = [
                worker.slot
                for worker in self.workers
                if held.get(worker.proc.pid if worker.proc is not None else -1, 0) < GPU_MIN_USED_MIB
            ]
            if not held or not pending or time.time() >= deadline:
                break  # no compute-apps view at all: go straight to the log fallback
            time.sleep(5.0)  # nvidia-smi lags context creation by a moment
        if held:
            if pending:
                raise RuntimeError(
                    f"worker(s) {pending} hold less than {GPU_MIN_USED_MIB} MiB on a GPU "
                    f"(nvidia-smi compute apps: {held!r}): the weights are not on the card, so the "
                    "rung stays disabled. Check that the CUDA release is the one in use, that "
                    "LD_LIBRARY_PATH carries the cudart directory, and read the worker log."
                )
            self.gpu_mib = {
                worker.slot: held.get(worker.proc.pid, 0)
                for worker in self.workers
                if worker.proc is not None
            }
            evidence = ", ".join(f"slot {slot}={mib} MiB" for slot, mib in sorted(self.gpu_mib.items()))
            log(f"gpu proof    : resident GPU memory per worker: {evidence}")
            return
        # nvidia-smi would not enumerate compute processes: fall back to the load log.
        for worker in self.workers:
            layers = gpu_layers_in_log(worker.log_path)
            if layers is None or layers[0] <= 0:
                raise RuntimeError(
                    f"cannot prove worker {worker.slot} put layers on the GPU: nvidia-smi lists no "
                    f"compute processes and {worker.log_path} has no offload line ({layers!r}). "
                    "Refusing to enable the rung on an unproven load."
                )
            log(f"gpu proof    : worker {worker.slot} log reports {layers[0]}/{layers[1]} layers offloaded")

    def _prove_reachable(self) -> None:
        """Prove the published port answers over the tailnet before Plexus is told about it."""
        ok, detail = self.ts.probe(self.ip, self.serve_port)
        if ok:
            log(f"tailnet      : reachability probe OK ({detail})")
            return
        log(f"tailnet      : reachability probe FAILED ({detail})")
        if os.environ.get("CLEF_ALLOW_UNREACHABLE", "").strip().lower() in ("1", "true", "yes", "on"):
            log("tailnet      : CLEF_ALLOW_UNREACHABLE set — registering without a reachability proof")
            return
        raise RuntimeError(
            f"tcp/{self.serve_port} does not answer over the tailnet ({detail}). Plexus would be "
            "pointed at a rung it cannot reach, so bring-up stops here. Check `tailscale serve "
            "status`, BackendState and the tag:endpoints ACL; set CLEF_ALLOW_UNREACHABLE=1 to skip "
            "this gate deliberately."
        )

    def _start_tailnet(self) -> None:
        # No cleanup pass and no shared hostname: this run's node name is its own UUID, so there is
        # nothing to delete and no peer to disturb.
        self.ts.start_daemon()
        authkey = self.ts.mint_auth_key()
        try:
            self.ts.up(authkey)
        finally:
            self.ts.revoke_auth_key()
        self.ip = self.ts.ipv4()
        self.dnsname = self.ts.dnsname()
        self.device_id = self.ts.own_device_id()
        # Userspace networking => every tailnet destination needs tailscaled's local proxy.
        # Without this the registrar call has no route and stalls or errors.
        os.environ["CLEF_TAILNET_PROXY"] = "http://127.0.0.1:1055"
        log(f"tailnet      : {self.ip} instance={RUN_ID} name={self.dnsname or '(no DNSName)'}")
        log("tailnet      : outbound proxy 127.0.0.1:1055 armed for tailnet destinations")
        if self.device_id:
            log(f"tailnet      : device id {self.device_id} is the only device teardown may delete")
        else:
            log(
                "WARNING: could not resolve this node's Tailscale device id — teardown cannot delete "
                "it; the ephemeral node expires on its own and no peer is affected"
            )

    def _start_proxy(self) -> None:
        last_error = None
        for port in SERVE_PORT_LADDER:
            proxy = RoundRobinProxy(self.workers, port)
            try:
                proxy.start()
            except OSError as exc:
                last_error = exc
                log(f"proxy        : 127.0.0.1:{port} unavailable ({exc}); next candidate")
                continue
            try:
                self.ts.serve_tcp(port)
            except Exception as exc:
                last_error = exc
                log(f"tailscale    : serve --tcp={port} failed ({exc}); next candidate")
                proxy.stop()
                continue
            self.proxy = proxy
            self.serve_port = port
            log(f"proxy        : 127.0.0.1:{port} round-robin over {[w.port for w in self.workers]}")
            log(f"tailscale    : serve tcp {port} -> 127.0.0.1:{port}")
            return
        raise RuntimeError(f"no usable serve port in {list(SERVE_PORT_LADDER)}: {last_error!r}")

    def _prove_plexus(self) -> None:
        """Prove this VM can reach the registrar over the tailnet before asking it for a lease.

        The `/health` probe above only proves the local tailnet stack + serve path
        (container -> itself). This is the first genuinely cross-node call, so failing here names
        the problem instead of surfacing it later as a mystery lease error.
        """
        started = time.time()
        request = urllib.request.Request(REGISTRY_URL + "/health")
        try:
            with _opener().open(request, timeout=PROBE_TIMEOUT) as response:
                body = json.load(response)
        except Exception as exc:
            raise RuntimeError(
                f"registrar unreachable over the tailnet after {time.time() - started:.1f}s "
                f"({type(exc).__name__}: {exc}); CLEF_REGISTRY_URL={REGISTRY_URL!r} "
                f"CLEF_TAILNET_PROXY={os.environ.get('CLEF_TAILNET_PROXY')!r}"
            ) from exc
        if not isinstance(body, dict) or body.get("status") != "ok":
            raise RuntimeError(f"registrar health was not ok: {body!r:.200}")
        log(f"registrar    : reachable over tailnet in {time.time() - started:.1f}s")

    def _register_provider(self) -> None:
        """Create this run's lease. The registrar, not this notebook, writes the provider."""
        self.base_url = f"http://{self.ip}:{self.serve_port}/v1"
        self.concurrency = len(self.workers)
        self.provider_timeout_ms = PROVIDER_TIMEOUT_MS
        self.lease = Lease(
            REGISTRY_URL,
            SECRETS["CLEF_REGISTRY_KEY"],
            IDENT,
            _opener,
            lambda: sum(1 for worker in self.workers if worker.alive() and worker.healthy()),
        )
        created = self.lease.create(self.base_url, self.concurrency, SECRETS["CLEF_REMOTE_KEY"])
        log(
            f"lease        : created {RUN_ID} workers={self.concurrency} "
            f"provider={created.get('provider')} phase={created.get('phase')}"
        )

    def _enable_rung(self) -> None:
        # There is no notebook-side enable step. The registrar owns clef-flash-notebooks and enables
        # exactly one target as part of admission; this method exists only for lifecycle compatibility.
        if self.lease is None:
            raise RuntimeError("lease not created")

    # ----------------------------------------------------------------- self-test
    def _self_test(self) -> None:
        """Admission: the registrar proves this provider through Plexus before publishing it.

        There is no local shortcut and no environment switch that admits this node unverified —
        an unproven target in the pool is exactly the failure v10 exists to prevent.
        """
        result = self.lease.ready()
        if result.get("provider") != PROVIDER or result.get("run_id") != RUN_ID:
            raise SelfTestFailure(
                f"admission response belongs to another runtime: {result!r:.300}"
            )
        self._rung_on = True
        log(f"admission    : {PROVIDER} admitted to clef-flash-notebooks ({result.get('phase')})")

    # ----------------------------------------------------------------- heartbeat
    def _watch(self) -> None:
        ts_misses = 0
        worker_misses = 0
        while not self._stop.wait(WATCH_INTERVAL):
            try:
                state = self.ts.backend_state()
            except Exception as exc:
                state = f"error:{exc!r}"
            if state == "Running":
                ts_misses = 0
            else:
                ts_misses += 1
                log(f"heartbeat    : BackendState={state!r} ({ts_misses}/2)")
                if ts_misses >= 2:
                    log("heartbeat    : tailnet is down — withdrawing the lease so the pool stops routing here")
                    self.stop(reason=f"tailscale BackendState={state}")
                    return
            # One healthy worker still serves; every worker dead or unhealthy twice running does not.
            # The lease's own heartbeat reports the healthy count, and the registrar shrinks this
            # provider's cap to match (or withdraws it at zero).
            states = [(worker.slot, worker.alive(), worker.healthy()) for worker in self.workers]
            if states and not any(healthy for _, _, healthy in states):
                worker_misses += 1
                log(f"heartbeat    : no healthy worker {states} ({worker_misses}/2)")
                if worker_misses >= 2:
                    log("heartbeat    : withdrawing the lease before stopping the node")
                    self.stop(reason="every worker is down or unhealthy")
                    return
            else:
                worker_misses = 0
            if self.lease is None:
                continue
            if self.lease.withdrawn:
                log("heartbeat    : lease is withdrawn — this runtime no longer serves; stopping")
                self.stop(reason="lease withdrawn")
                return
            if self.lease.lost:
                # Renewal has not been confirmed for a while. Do NOT re-advertise by hand and do not
                # assume the pool still contains this target: ask the registrar what it thinks.
                try:
                    phase = str(self.lease.status().get("phase"))
                except Exception as exc:
                    log(f"heartbeat    : lease unconfirmed and status unreadable ({exc}); retrying")
                    continue
                if phase in ("withdrawn", "expired"):
                    log(f"heartbeat    : lease {phase} — pool exclusion confirmed; stopping this runtime")
                    self.stop(reason=f"lease {phase}")
                    return
                log(f"heartbeat    : lease renewal unconfirmed (phase={phase}); not re-advertising")

    # ------------------------------------------------------------------ teardown
    def _arm_callbacks(self) -> None:
        if self._armed:
            return
        self._armed = True
        atexit.register(self.stop)
        for name in ("SIGINT", "SIGTERM", "SIGHUP"):
            sig = getattr(signal, name, None)
            if sig is None:
                continue
            try:
                previous = signal.getsignal(sig)
            except Exception:
                continue

            def handler(signum, frame, _sig=sig, _prev=previous):
                try:
                    signal.signal(_sig, _prev)  # second Ctrl-C interrupts for real
                except Exception:
                    pass
                threading.Thread(
                    target=self.stop, kwargs={"reason": f"signal {signum}"}, daemon=True
                ).start()

            try:
                signal.signal(sig, handler)
            except (ValueError, OSError):
                pass  # not the main thread, or unsupported



    def _await_teardown(self) -> None:
        """Wait for an in-flight teardown instead of returning while sockets are still closing.

        atexit and the signal handlers can both land here while the heartbeat thread is already
        tearing down; returning early lets the interpreter exit between "rung disabled" and
        "tailscale down", which leaves the daemon joined to no purpose.
        """
        if self._done.wait(timeout=300):
            return
        log("WARNING: teardown is still running after 300 s; returning without waiting further")

    def stop(self, reason: str = "", force: bool = False) -> None:
        with self._guard:
            if self._stopped:
                self._await_teardown()
                return
            self._stopped = True
            self._done.clear()
        log(f"teardown     : start{' — ' + reason if reason else ''}")
        leasing = self.lease is not None
        withdrawn = not leasing
        if leasing:
            # 1. Lease first: the registrar must stop selecting this node before its sockets die.
            # If that cannot be verified, do not create a routed dead endpoint.
            for attempt in range(3):
                try:
                    result = self.lease.withdraw()
                    if result.get("phase") == "withdrawn":
                        withdrawn = True
                        log(f"lease        : {RUN_ID} withdrawn (re-GET verified)")
                        break
                    log(f"lease        : withdraw attempt {attempt + 1}/3 returned {result!r:.200}")
                except Exception as exc:
                    log(f"lease        : withdraw attempt {attempt + 1}/3 failed ({exc})")
                time.sleep(2.0)
            if not withdrawn and not force:
                log(
                    "teardown     : INCOMPLETE — lease withdrawal is unconfirmed, so workers, proxy "
                    "and tailnet are being LEFT RUNNING to avoid a routed dead endpoint. Fix the "
                    "registrar/Plexus error and re-run stop(), or call stop(force=True) to destroy "
                    "the node anyway."
                )
                with self._guard:
                    self._stopped = False
                self._done.set()
                return
            if not withdrawn:
                log(
                    "WARNING: stop(force=True) is destroying the node while its pool target may still "
                    f"be enabled: clef-flash-notebooks -> clef-nb-{RUN_ID}"
                )
        self._stop.set()
        try:
            if withdrawn and leasing:
                # 2. Let Plexus reload routing, then let local in-flight requests finish. A request
                # already inside the proxy is still someone's answer; killing it mid-flight turns a
                # clean stop into an error on the caller's side.
                time.sleep(10.0)
                deadline = time.time() + 600.0
                while (
                    self.proxy is not None
                    and getattr(self.proxy, "active", 0) > 0
                    and time.time() < deadline
                ):
                    log(f"teardown     : draining, active={self.proxy.active}")
                    time.sleep(2.0)
                if self.proxy is not None and getattr(self.proxy, "active", 0) > 0:
                    log(
                        f"teardown     : INCOMPLETE — {self.proxy.active} request(s) still in flight "
                        "after the 600 s drain deadline; workers are being left running"
                    )
                    if not force:
                        with self._guard:
                            self._stopped = False
                        self._done.set()
                        return
            # 3. workers
            for worker in self.workers:
                log(f"worker {worker.slot}     : stopping")
                try:
                    worker.stop()
                except Exception as exc:
                    log(f"worker {worker.slot}     : stop failed ({exc})")
            # 4. proxy
            if self.proxy is not None:
                try:
                    self.proxy.stop()
                    log("proxy        : closed")
                except Exception as exc:
                    log(f"proxy        : stop failed ({exc})")
            # 5. tailnet: resolve and delete ONLY this run's device, while status still has Self.ID.
            if self.ts is not None:
                device_id = None
                try:
                    device_id = self.ts.own_device_id()
                except Exception as exc:
                    log(f"tailscale    : own device id unreadable ({exc})")
                if device_id:
                    try:
                        if self.ts.delete_device(device_id):
                            log(f"tailscale    : own device {device_id} deleted (ownership verified)")
                    except Exception as exc:
                        log(
                            f"WARNING: own device deletion not confirmed ({exc}); the ephemeral node "
                            "expires on its own and no other device is touched"
                        )
                else:
                    log("WARNING: own device id unresolved — leaving the ephemeral node to expiry")
                try:
                    self.ts.serve_reset()
                except Exception as exc:
                    log(f"tailscale    : serve reset failed ({exc})")
                try:
                    self.ts.down()
                    log("tailscale    : node left the tailnet")
                except Exception as exc:
                    log(f"tailscale    : down failed ({exc})")
                try:
                    self.ts.stop_daemon()
                except Exception as exc:
                    log(f"tailscale    : daemon stop failed ({exc})")
        finally:
            if self._vm_lock is not None:
                try:
                    fcntl.flock(self._vm_lock.fileno(), fcntl.LOCK_UN)
                    self._vm_lock.close()
                except Exception:
                    pass
                self._vm_lock = None
            self._done.set()
            log("teardown     : done")

    def wait(self) -> None:
        log("runtime      : holding open — Ctrl-C for a graceful shutdown")
        try:
            while not self._stop.wait(1.0):
                pass
        except KeyboardInterrupt:
            log("runtime      : interrupt")
            self.stop(reason="KeyboardInterrupt")
            return
        self._await_teardown()

    # -------------------------------------------------------------------- status
    def status(self) -> None:
        lines = []
        gpus = ", ".join(f"cuda:{gpu['index']} {gpu['name']} {gpu['gib']:.2f} GiB" for gpu in GPUS)
        lines.append(f"gpus        : {len(GPUS)} — {gpus}")
        for worker in self.workers:
            pid = worker.proc.pid if worker.proc is not None else "-"
            kv = worker.ctx * KV_BYTES_PER_TOKEN / 2**30
            lines.append(
                f"worker {worker.slot}     : cuda:{worker.cuda} port={worker.port} ctx={worker.ctx} "
                f"kv={kv:.2f} GiB pid={pid} alive={worker.alive()}"
            )
        if self.gpu_mib:
            evidence = ", ".join(f"slot {slot}={mib} MiB" for slot, mib in sorted(self.gpu_mib.items()))
            lines.append(f"gpu proof   : {evidence}")
        if self.proxy is not None:
            lines.append(
                f"proxy       : 127.0.0.1:{self.proxy.port} served={self.proxy.served} "
                f"failures={self.proxy.failures}"
            )
        lines.append(f"tailnet     : {self.ip or '-'}  {self.dnsname or '-'}")
        lines.append(f"serve       : tcp {self.serve_port} -> 127.0.0.1:{self.serve_port}")
        lines.append(
            f"provider    : {PROVIDER} systemone -> {self.base_url or '-'} "
            f"maxConcurrency={self.concurrency or '-'} timeoutMs={self.provider_timeout_ms or '-'}"
        )
        lines.append(f"rung        : {self.client.describe_rung()}")
        print("=" * 78)
        for line in lines:
            print(line)
        print("=" * 78, flush=True)


In [ ]:
# [11/13] bring the node up and register it. Prints the status block on success.
import json as _json_status
import pathlib as _path_status
import traceback as _tb_status


def _status_file() -> _path_status.Path:
    """RUN_STATUS.json next to the session's outputs.

    Kaggle uploads /kaggle/working as kernel output, so a hard kill still leaves the
    status there.  Colab has no /kaggle at all -- writing to that path raised
    FileNotFoundError INSIDE the except handler, which replaced the real bring-up
    error with a missing-directory error.
    """
    base = _path_status.Path("/kaggle/working")
    if not base.is_dir():
        base = _path_status.Path.cwd()
    return base / "RUN_STATUS.json"


def _write_status(payload: dict) -> None:
    """Best effort: a status write must never become the failure it is reporting."""
    target = _status_file()
    try:
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(_json_status.dumps(payload, indent=2))
        print(f"status       : {target}", flush=True)
    except Exception as exc:  # noqa: BLE001
        print(f"status write failed at {target}: {exc!r}", flush=True)


try:
    _prior = globals().get("RUNTIME")
    if _prior is not None and not getattr(_prior, "_stopped", False):
        raise RuntimeError(
            "a live Runtime already exists in this kernel — run the shutdown cell ([13/13]) "
            "first. Starting another would orphan its workers, proxy and lease."
        )
    RUNTIME = Runtime()
    RUNTIME.start()
except Exception:
    _tb_status.print_exc()
    _write_status(
        {
            "phase": "bringup",
            "ok": False,
            "run_id": RUN_ID,
            "provider": PROVIDER,
            "pool": "clef-flash-notebooks",
            "error": _tb_status.format_exc(),
        }
    )
    raise
else:
    _write_status(
        {
            "phase": "serving",
            "ok": True,
            "run_id": RUN_ID,
            "provider": PROVIDER,
            "pool": "clef-flash-notebooks",
        }
    )


In [ ]:
# [12/13] hold the runtime open. Keep the Kaggle/Colab session open; stop it from the UI when done.
# Ctrl-C here disables the rung, kills the workers and leaves the tailnet.
RUNTIME.wait()

In [ ]:
# [13/13] explicit graceful shutdown — idempotent, safe to run twice.
# Order: lease withdrawn (verified) -> routing reload + local drain -> workers -> proxy
#        -> this run's own tailnet node.
RUNTIME.stop(reason="manual shutdown cell")
try:
    print("lease now:", RUNTIME.lease.status() if RUNTIME.lease is not None else "no lease")
except Exception as exc:
    print("lease state unreadable:", exc)


## Wire contract (what Plexus sends, what llama.cpp must answer)

Plexus POSTs to `{provider.api_base_url.systemone}/systemone` with
`Authorization: Bearer <that provider's api_key>` and a `decisions` body:

```json
{"model": "clef-flash",
 "state": "<string | object | array>",
 "questions": {
   "<id>": {"type": "noul", "instructions": "<string | object | array>"},
   "<id>": {"type": "choice", "instructions": "...", "criteria": {"<option>": "<description | null>"}},
   "<id>": {"type": "score", "instructions": "...", "criteria": ["<level description>", "..."]}
 }}
```

It requires at least this back:

```json
{"model": "<string>",
 "answers": {
   "<id>": {"type": "choice", "choice": "<option>", "probabilities": {}, "confidence": 0.0},
   "<id>": {"type": "score", "score": 0.0, "legend": {}, "probabilities": {}},
   "<id>": {"type": "noul", "noul": 0.0}
 },
 "usage": {"input_tokens": 0, "output_tokens": 0}}
```

`llama-server` implements exactly this at `POST /v1/systemone`: it reads the GGUF's `systemone`
chat template and the decision head, and it does **not** validate the `model` field — so the routed
model string stays `clef-flash`. A real call through Plexus returned choice/score/noul answers
with `usage {"input_tokens": 335, "output_tokens": 0}`.

The proxy passes path and body through unchanged, so the contract is end-to-end: proxy auth
(`Bearer <this run's bearer>`) → worker auth (same value) → llama.cpp `/v1/systemone`.

**Score scales.** A `score` question returns `0 .. len(criteria)-1`, fractional; a `noul` returns
`0 .. 1`. `confidence` is provider-specific and is **not** a portable correctness guarantee — the
registrar's admission probe checks typed transport and capability, not model calibration, and it
never applies a confidence threshold.

**Timeouts.** The registrar creates each provider with `timeoutMs=600000` (10 min); the proxy's
upstream timeout is 540 s, so its own 502 reaches Plexus before Plexus abandons the request. Ten
minutes is still not a promise for large prompts: pre-fill on one T4 runs in the low hundreds of
tokens per second, so a ~100k-token `state` costs minutes of pre-fill before the first token.
Keep `state` inside the worker's context window and measure `usage.input_tokens` on a real call
before routing long prompts here.


## Verifying from another host

From a tailnet peer. `100.x.y.z` is this node's tailnet IPv4 and `clef-nb-<uuid>` is the provider
slug, both printed by the status block in the bring-up cell.

```bash
PLEXUS_BASE=https://<your-plexus-base>    # your gateway base URL; export it, never commit it
export PLEXUS_BASE

# 1. node liveness (unauthenticated by design: run id, per-worker health, in-flight count)
curl -s http://100.x.y.z:8080/health | jq

# 2. the proxy refuses unauthenticated inference
curl -s -o /dev/null -w '%{http_code}\n' -X POST http://100.x.y.z:8080/v1/systemone -d '{}'
# -> 401

# 3. lease state straight from the registrar (bootstrap key, not a Plexus credential)
curl -s http://100.x.y.z:8769/v1/leases/<uuid> \
  -H "authorization: Bearer $CLEF_REGISTRY_KEY" -H "x-lease-token: <lease token>" | jq

# 4. what the registrar stored for THIS run's provider
curl -s -H "x-admin-key: $PLEXUS_ADMIN_KEY" \
  $PLEXUS_BASE/v0/management/providers/clef-nb-<uuid> | jq '.api_base_url, .maxConcurrency'

# 5. the dedicated pool: this run's target, enabled exactly once
curl -s -H "x-admin-key: $PLEXUS_ADMIN_KEY" \
  $PLEXUS_BASE/v0/management/aliases/clef-flash-notebooks \
  | jq '.target_groups[] | {name, selector, targets}'

# 6. end-to-end through Plexus, scoped to this one provider
curl -s $PLEXUS_BASE/v1/systemone \
  -H "Authorization: Bearer <a key scoped to clef-nb-<uuid>>" -H 'content-type: application/json' \
  -d '{"model":"direct/clef-nb-<uuid>/clef-flash","state":"ping","questions":{"n":{"type":"noul","instructions":"hi"}}}' | jq
```

Inside the VM the status block is authoritative: worker `n_ctx` is read back from `/props`, the
`/health` body carries this run's `run_id`, and admission is proven by the registrar rather than
by a local probe.


## Direct provider calls

Plexus resolves `direct/<provider>/<model>` before the alias selector, so a direct call proves one
provider rather than the whole fallback chain. The registrar uses exactly this shape for admission,
with a temporary inference key whose `allowedProviders` is `["clef-nb-<uuid>"]` — so no other
provider can answer in its place, and the key is deleted (and its absence verified) immediately
afterwards.

A notebook holds no inference key at all, so operator direct calls need a key you scope yourself:

```bash
PLEXUS_BASE=https://<your-plexus-base>    # your gateway base URL; export it, never commit it
export PLEXUS_BASE

# this runtime's provider, bypassing the pool alias
curl -s $PLEXUS_BASE/v1/systemone \
  -H "Authorization: Bearer <key scoped to clef-nb-<uuid>>" -H 'content-type: application/json' \
  -d '{"model":"direct/clef-nb-<uuid>/clef-flash","state":"ping","questions":{"n":{"type":"noul","instructions":"hi"}}}' | jq
```

Mint a throwaway key for the check and delete it afterwards — do not broaden a shared key. Build the
body with `jq` and keep the secret in a shell variable so neither ever appears in a document:

```bash
THROWAWAY="sk-$(openssl rand -hex 24)"     # local only; never paste it into a notebook
jq -n --arg s "$THROWAWAY" --arg p "clef-nb-<uuid>" \
  '{secret:$s, comment:"temporary, delete after use", allowedProviders:[$p]}' > /tmp/probe-key.json
curl -s -X PUT "$PLEXUS_BASE/v0/management/keys/zz-verify-<uuid>" \
  -H "x-admin-key: $PLEXUS_ADMIN_KEY" -H 'content-type: application/json' \
  --data @/tmp/probe-key.json
# ... use it ...
curl -s -X DELETE "$PLEXUS_BASE/v0/management/keys/zz-verify-<uuid>" \
  -H "x-admin-key: $PLEXUS_ADMIN_KEY"
shred -u /tmp/probe-key.json 2>/dev/null || rm -f /tmp/probe-key.json
```

The old `clef-llama-colab-v2` key allows only `clef-llama-remote` and `clef-flash-mac`; a uniquely
named provider is refused by it rather than silently rerouted. That key policy is deliberately
untouched by v10.


## Troubleshooting

**`another kernel on this VM holds .../clef-runtime.lock`.** One runtime per VM is by design: a
second kernel would contend for the GPUs, ports 8091/8092, the proxy port and tailscaled's
userspace proxy on 1055. Run the shutdown cell ([13/13]) in the other kernel first. Independent
Colab or Kaggle runtimes are unaffected — they are separate VMs.

**`a live Runtime already exists in this kernel`.** The configuration cell refuses to rebuild
globals under a running runtime, because the old workers, sockets and lease would be orphaned.
Stop it first.

**Worker dies during load.** `RuntimeError: worker N exited rc=...` carries the tail of
`runs/<uuid>/logs/llama-N.log`. CUDA OOM on a 15 GiB T4 means the context is too greedy: re-run
with `CLEF_CTX=16384` and bring up again.

**`worker(s) [0] hold less than 2048 MiB on a GPU`.** Bring-up proved the weights are not on the
card, so admission never happens. Either the wrong release was unpacked (no `libggml-cuda.so`), or
`LD_LIBRARY_PATH` is missing the `cudart-*` directory, or CUDA init failed and llama.cpp loaded on
the CPU. The worker log names the cause.

**`tcp/8080 does not answer over the tailnet`.** The probe left through tailscaled's userspace
SOCKS5 proxy and came back through `tailscale serve`; it failed, so the registrar was never asked
to admit this node. Check `tailscale serve status`, that `BackendState` is `Running`, and that the
ACL lets `tag:endpoints` traffic reach the port. `CLEF_ALLOW_UNREACHABLE=1` skips the gate and
leaves reachability unproven.

**`registrar unreachable over the tailnet`.** `CLEF_REGISTRY_URL` must be the tailnet URL of the
registrar (e.g. `http://100.x.y.z:8769`), and the VM only leaves via tailscaled's proxy — which is
armed by then. Prefer the literal tailnet IPv4: MagicDNS may be off on the peer.

**Admission refused (`ready` returns 4xx/5xx).** The registrar's probe is a real System One call
with a temporary key scoped to this provider. A 409 means a probe is already in flight, a 410 that
the lease ended (start a new runtime), a 503 that its Plexus write could not be confirmed. Read
the message; do not work around it by enabling anything by hand.

**`teardown : INCOMPLETE`.** Withdrawal was not confirmed, so workers, proxy and tailnet were left
running on purpose. Fix the registrar/Plexus error and re-run `stop()`, or accept the routed dead
endpoint only via `stop(force=True)`.

**Hard death (VM deleted / evicted).** No teardown runs. The registrar expires the lease within
180 s and disables that target; the provider and its tombstone stay, so the slug is never reused.
No manual pool edit is needed, and no other runtime is touched.

**Plexus gets 401 from this node.** The registrar wrote the provider's `api_key` from the lease
itself, so a mismatch means someone edited the provider. Do not fix it by editing the notebook —
withdraw the lease and start a new runtime.

**`tailscale up` prints a login URL instead of joining.** The minted key was rejected, or
`TAILSCALE_API_KEY` lacks device/key write authority. Check the token's scope.

**ACL blocks traffic.** The Plexus host must reach this node on the serve port (8080 or 8443), and
this node must reach the registrar on 8769. In userspace mode the daemon still terminates inbound
connections to served ports.

**Port ladder.** If 8080 is occupied inside the VM the proxy binds 8443, `tailscale serve` is
published on 8443, and the lease carries the 8443 base URL — no manual step.
